In [0]:
%sql
-----------------------------------------
--- VIEWERSHIP
-----------------------------------------
--- Checking if the table is properly uploaded
SELECT * 
FROM brighttv.case.viewership;
--- counting the number of rows
SELECT COUNT(*) AS total_rows
FROM brighttv.case.viewership;
--- checking duplicates
SELECT *,
       COUNT(*) AS duplicate_count
FROM brighttv.case.viewership
GROUP BY ALL
HAVING COUNT(*) > 1;
--- counting the number of unique viewers
SELECT COUNT(DISTINCT UserID0) AS unique_viewers
FROM brighttv.case.viewership;

SELECT COUNT (DISTINCT userid4) AS unique_viewers
FROM brighttv.case.viewership;
--- Converting timestamp from UTC to SAST
SELECT
    *,
    from_utc_timestamp(
        RecordDate2,
        'Africa/Johannesburg'
    ) AS RecordDate3_SAST
FROM brighttv.case.viewership;
--- saving converted timestamp
CREATE OR REPLACE TABLE brighttv.case.viewership AS
SELECT
    * EXCEPT (RecordDate3_SAST),
    from_utc_timestamp(
        RecordDate2,
        'Africa/Johannesburg'
    ) AS RecordDate3_SAST
FROM brighttv.case.viewership;
--- checking if the changes are saved
SELECT
    RecordDate2 AS UTC_Time,
    RecordDate3_SAST AS SAST_Time
FROM brighttv.case.viewership
LIMIT 20;
--- counting the number of distinct channels
SELECT COUNT(DISTINCT Channel2) AS unique_channels
FROM brighttv.case.viewership;

-------------------------------------------
--- USER PROFILES
-------------------------------------------
--- checking if the table is successfully uploaded
SELECT *
FROM brighttv.case.user_profiles;
--- checking duplicates
SELECT *,
       COUNT(*) AS duplicate_count
FROM brighttv.case.user_profiles
GROUP BY ALL
HAVING COUNT(*) > 1;
--- counting the number of unique users
SELECT COUNT(DISTINCT UserID) AS unique_users
FROM brighttv.case.user_profiles;

-----------------------------------------------
--BIG CODE
-----------------------------------------------
WITH CTE_1 AS (
    SELECT UserID,
           Name,
           Surname,
--- check the email column and replacing blanks
           CASE
               WHEN Email = 'None' THEN 'Unknown'
               WHEN Email = '' THEN 'Unknown'
               ELSE Email
           END AS Email,
--- checking the province column and replacing blanks
           CASE
               WHEN Province = 'None' THEN 'Unknown'
               WHEN Province = '' THEN 'Unknown'
               ELSE Province
           END AS Province,
--- checking the gender colunmn and replacing blanks
           CASE
               WHEN Gender = 'None' THEN 'Unknown'
               WHEN GENDER = '' THEN 'Unknown'
               ELSE Gender
           END AS Sex,
--- checking the age column and replacing blanks
         CASE
              WHEN Age = 0 THEN 'Infant'
              WHEN Age BETWEEN 1 AND 12 THEN 'Child'
              WHEN Age BETWEEN 13 AND 19 THEN 'Teenager'
              WHEN Age BETWEEN 20 AND 35 THEN 'Young Adult'
              WHEN Age BETWEEN 36 AND 50 THEN 'Adult'
              WHEN Age BETWEEN 51 AND 60 THEN 'Elder'
              WHEN Age > 60 THEN 'Senior citizen'
              ELSE 'Unknown'
        END AS Age,
--- checking race column and replacing blanks
        CASE
              WHEN Race = 'None' THEN 'Unknown'
              WHEN Race = '' THEN 'Unknown'
              WHEN Race = 'other' THEN 'Unknown'
              ELSE Race 
          END AS Ethnicity

FROM brighttv.case.user_profiles
),

CTE_2 AS (
SELECT 
COALESCE (UserID0, Userid4) AS UserID, --- Combining the two UserID columns into one
--- checking the channel2 column and replacting blanks
CASE
    WHEN Channel2 = 'None' THEN 'Unknown'
    WHEN Channel2 = '' THEN 'Unknown'
    ELSE Channel2
END AS tv_channels,
--- DATE FUNCTIONS 
DAYOFWEEK(TO_DATE(RecordDate3_SAST)) AS number_of_day_of_week, --- finding the number of the day the week from RecordDate3_SAST
DAYNAME(TO_DATE(RecordDate3_SAST)) AS day_name,---- finding the name of the day of the week
DATE_FORMAT(RecordDate3_SAST,'MMMM') AS month_name,---- finding the name of the month
DATE_FORMAT(RecordDate3_SAST, 'yyyyMM') AS month_id, --- finding the month id
TO_DATE(RecordDate3_SAST) AS actual_date,--- finding the actual date
DATE_FORMAT(RecordDate3_SAST,'HH:mm:ss') AS watch_time, --- finding the time of the day

--- CASE STATEMENTS
--- classifying the days of the week
CASE 
     WHEN DAYNAME(TO_DATE(RecordDate3_SAST)) IN ('Sat', 'Sun') THEN 'Weekend'
     ELSE 'Weekday'
     END AS day_classification,
--- classifying watch time buckets
CASE
    WHEN watch_time BETWEEN '00:00:00' AND '06:00:00' THEN 'Early morning'
    WHEN watch_time BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
    WHEN watch_time BETWEEN '12:00:00' AND '17:59:59' THEN 'Afternoon'
    WHEN watch_time BETWEEN '18:00:00' AND '23:59:59' THEN 'Evening'
    ELSE 'Unknown'
    END AS watch_time_classification,
--- classifying viewing span
CASE
    WHEN CAST(Duration_2 AS INTERVAL HOUR TO SECOND) = INTERVAL '0:00:00' HOUR TO SECOND THEN 'No view'
    WHEN CAST(Duration_2 AS INTERVAL HOUR TO SECOND) BETWEEN INTERVAL '0:00:01' HOUR TO SECOND AND INTERVAL '0:09:00' HOUR TO SECOND THEN 'Short span'
    WHEN CAST(Duration_2 AS INTERVAL HOUR TO SECOND) BETWEEN INTERVAL '0:10:00' HOUR TO SECOND AND INTERVAL '0:29:00' HOUR TO SECOND THEN 'Medium span'
    WHEN CAST(Duration_2 AS INTERVAL HOUR TO SECOND) BETWEEN INTERVAL '0:30:00' HOUR TO SECOND AND INTERVAL '0:59:00' HOUR TO SECOND THEN 'Long span'
    WHEN CAST(Duration_2 AS INTERVAL HOUR TO SECOND) > INTERVAL '0:59:00' HOUR TO SECOND THEN 'Very long span'
END AS Viewing_Time,

---converting duration into minutes
 ROUND(
        CAST(SPLIT(Duration_2, ':')[0] AS INT) * 60      -- hours to minutes
      + CAST(SPLIT(Duration_2, ':')[1] AS INT)           -- minutes
      + CAST(SPLIT(Duration_2, ':')[2] AS INT) / 60.0,   -- seconds to minutes
    2) AS Duration_Minutes

FROM brighttv.case.viewership
)
--- JOINING the two tables
SELECT 
     CTE_1.Name,
     CTE_1.Surname,
     CTE_1.Email,
     CTE_1.Province,
     CTE_1.Age,
     CTE_1.Sex,
     CTE_1.Ethnicity,
     CTE_2.tv_channels,
     CTE_2.UserID,
     CTE_2.number_of_day_of_week, 
     CTE_2.day_name,
     CTE_2.month_name,
     CTE_2.month_id,
     CTE_2.actual_date,
     CTE_2.day_classification,
     CTE_2.Viewing_Time,
     CTE_2.Duration_Minutes,
     CTE_2.watch_time
FROM CTE_2
INNER JOIN CTE_1
ON CTE_1.UserID = CTE_2.UserID
ORDER BY COALESCE(CTE_1.UserID, CTE_2.UserID), tv_channels;



